# EEG QC — pipeline chính thức (notch + bandpass 1-45Hz + segment rejection)

**Pipeline chính thức cho phân tích spectral/band power.** Quyết định dùng
bản tối giản này thay vì HAPPE đầy đủ (có wavelet) vì trên dữ liệu 2 kênh, wavelet xoá phần lớn công
suất phổ ở mọi băng tần (kể cả alpha)
mà chỉ tăng nhẹ số epoch giữ lại — không đáng, nhất là khi
segment rejection (±150µV, đơn vị đã fix) tự đủ sức loại epoch hỏng.

Notebook này KHÔNG chạy qua `happe.run_pipeline()` — chỉ gọi trực tiếp 3 bước:

1. **Line noise**: notch thật (zero-phase), 50Hz + harmonic
2. **Bandpass**: 1-45Hz FIR
3. **Segment + reject**: epoch 2s cố định, loại theo biên độ ±150µV

Nội dung: chạy pipeline trên toàn bộ session, tổng quan QC, và so sánh
trước/sau áp dụng (PSD, công suất theo băng tần, coherence tại 50Hz) — không
so với pipeline HAPPE (xem `eeg_happe_qc.ipynb` nếu cần, giữ làm tham chiếu).

In [ ]:
from pathlib import Path
import re
import shutil
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

REPO_ROOT = Path("..").resolve()

LIB_DIR = REPO_ROOT / "lib"
HAPPE_SRC = REPO_ROOT / "happe-python" / "src"
for p in (LIB_DIR, HAPPE_SRC):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import mne
from happe.config import default_params
from happe.io import load_raw, save_outputs
from happe.steps import line_noise, filtering, segment
from happe.qc.metrics import assess_pipeline_step, line_noise_freqs_of_interest
from happe.exceptions import AllTrialsRejectedError, SingleTrialRejectionError
import happe
mne.set_log_level("ERROR")
%matplotlib inline
print("happe:", happe.__file__, "| version:", happe.__version__)

## 1. Gom file `eeg_labeled_raw.fif` từ Layer 2

In [ ]:
LAYER2_DIR = REPO_ROOT / "data" / "analytics" / "layer2"
OUT_DIR = REPO_ROOT / "data" / "analytics" / "eeg_qc"
STAGING_IN_DIR = OUT_DIR / "_staging_input"
EPOCHS_DIR = OUT_DIR / "epochs_processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
if STAGING_IN_DIR.exists():
    shutil.rmtree(STAGING_IN_DIR)
STAGING_IN_DIR.mkdir(parents=True)
if EPOCHS_DIR.exists():
    shutil.rmtree(EPOCHS_DIR)
EPOCHS_DIR.mkdir(parents=True)

PARTICIPANT_RE = re.compile(r"participant=([^/\\]+)")
STUDY_DAY_RE = re.compile(r"study_day=([^/\\]+)")
SESSION_RE = re.compile(r"session_id=([^/\\]+)")

fif_files = sorted(LAYER2_DIR.rglob("eeg_labeled_raw.fif"))
print(f"Tìm thấy {len(fif_files)} file eeg_labeled_raw.fif")

identity_by_stem = {}
for src in fif_files:
    s = str(src)
    participant = PARTICIPANT_RE.search(s).group(1)
    study_day = STUDY_DAY_RE.search(s).group(1)
    session_id = SESSION_RE.search(s).group(1)
    stem = f"{participant}_day{study_day}_{session_id[:8]}"
    dst = STAGING_IN_DIR / f"{stem}.fif"
    shutil.copyfile(src, dst)
    identity_by_stem[f"{stem}.fif"] = {
        "participant_key": participant, "study_day": int(study_day) if study_day.isdigit() else study_day,
        "session_id": session_id,
    }

identity_df = pd.DataFrame.from_dict(identity_by_stem, orient="index").rename_axis("File").reset_index()
print(f"Đã dàn {len(identity_by_stem)} file vào {STAGING_IN_DIR}")
identity_df.head()

## 2. Cấu hình

In [ ]:
MINIMAL_CONFIG = default_params(erp=False)
MINIMAL_CONFIG.lineNoise.freq = 50.0
MINIMAL_CONFIG.lineNoise.method = "notch"
MINIMAL_CONFIG.filt.highpass = 1.0
MINIMAL_CONFIG.filt.lowpass = 45.0
MINIMAL_CONFIG.segment.enabled = True
MINIMAL_CONFIG.segment.reg_len_s = 2.0
MINIMAL_CONFIG.segRej.enabled = True
MINIMAL_CONFIG.segRej.by_amplitude = True
MINIMAL_CONFIG.segRej.amp_min = -150e-6      # Volt = 150µV (đơn vị đã fix)
MINIMAL_CONFIG.segRej.amp_max = 150e-6
MINIMAL_CONFIG.outputFormat.formats = ["fif"]

MINIMAL_CONFIG.validate()
print(MINIMAL_CONFIG.to_yaml())

OUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUT_DIR / "run_config.yaml", "w") as f:
    f.write(MINIMAL_CONFIG.to_yaml())

## 3. Chạy trên toàn bộ session

Mỗi file: đo QC line-noise (coherence trước/sau notch) và công suất theo băng
tần (trước/sau notch+bandpass, trên tín hiệu liên tục) NGAY trong lúc xử lý —
tránh phải đọc lại file. Sau đó segment + reject, lưu epoch.

In [ ]:
BANDS = {"delta": (1, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 30), "gamma": (30, 45)}

def band_power(sig, srate):
    freqs, psd = welch(sig, fs=srate, nperseg=int(srate * 4))
    out = {}
    for name, (lo, hi) in BANDS.items():
        idx = (freqs >= lo) & (freqs <= hi)
        out[name] = np.trapz(psd[idx], freqs[idx])
    return out

def psd_curve(sig, srate):
    return welch(sig, fs=srate, nperseg=int(srate * 4))

def process_file_qc(path, params, epochs_dir):
    stem = Path(path).stem
    raw = load_raw(path, params)
    srate = raw.info["sfreq"]
    picks = mne.pick_types(raw.info, eeg=True)

    before = raw.get_data(picks=picks).copy()

    line_noise.apply(raw, params)
    foi = line_noise_freqs_of_interest(params.lineNoise.freq, params.lineNoise.harmonics,
                                       params.lineNoise.neighbors)
    after_notch = raw.get_data(picks=picks).copy()
    ln_qc = assess_pipeline_step("line_noise", before, after_notch, srate, foi)

    filtering.bandpass(raw, params)
    after_bp = raw.get_data(picks=picks).copy()

    bp_before = {k: np.mean([band_power(before[i], srate)[k] for i in range(before.shape[0])]) for k in BANDS}
    bp_after = {k: np.mean([band_power(after_bp[i], srate)[k] for i in range(after_bp.shape[0])]) for k in BANDS}

    epochs = segment.segment(raw, params)
    rej = segment.reject(epochs, params)
    epochs = rej.epochs
    save_outputs(epochs, epochs_dir, stem, params)

    data_row = {
        "File": Path(path).name,
        "File_Length_s": round(raw.n_times / srate, 3),
        "N_Epochs_Pre_Rejection": rej.n_pre,
        "N_Epochs_Post_Rejection": rej.n_post,
        "Pct_Epochs_Retained": round(rej.pct_retained, 2),
    }
    pipeline_row = {"File": Path(path).name}
    for k, v in ln_qc.items():
        pipeline_row[f"lineNoise_{k}"] = v
    for k in BANDS:
        pipeline_row[f"{k}_power_before"] = bp_before[k]
        pipeline_row[f"{k}_power_after"] = bp_after[k]
        pipeline_row[f"{k}_power_retained_pct"] = 100.0 * bp_after[k] / bp_before[k] if bp_before[k] > 0 else np.nan

    # PSD trung bình 2 kênh, trước (raw thô) và sau (notch+bandpass) -- dùng cho biểu đồ mục 5.
    f_before, psd_before = psd_curve(before.mean(axis=0), srate)
    f_after, psd_after = psd_curve(after_bp.mean(axis=0), srate)

    return data_row, pipeline_row, (f_before, psd_before, f_after, psd_after)


rows, pipeline_rows, errors, psd_examples = [], [], [], {}
files = sorted(STAGING_IN_DIR.iterdir())
for i, f in enumerate(files):
    try:
        data_row, pipeline_row, psd_curves = process_file_qc(str(f), MINIMAL_CONFIG, EPOCHS_DIR)
        rows.append(data_row)
        pipeline_rows.append(pipeline_row)
        psd_examples[f.stem] = psd_curves
        print(f"[{i+1}/{len(files)}] {f.name} ok")
    except (AllTrialsRejectedError, SingleTrialRejectionError) as exc:
        errors.append({"File": f.name, "Error": str(exc)})
        print(f"[{i+1}/{len(files)}] {f.name} LOI: {exc}")

data_qc = pd.DataFrame(rows)
pipeline_qc = pd.DataFrame(pipeline_rows)
error_log = pd.DataFrame(errors)
print(f"\nThanh cong: {len(data_qc)} | Loi: {len(error_log)}")
if len(error_log):
    print(error_log.to_string(index=False))

In [ ]:
data_qc = data_qc.merge(identity_df, on="File", how="left")
pipeline_qc = pipeline_qc.merge(identity_df, on="File", how="left")
cols_front = ["participant_key", "study_day", "session_id", "File"]
data_qc = data_qc[cols_front + [c for c in data_qc.columns if c not in cols_front]]
pipeline_qc = pipeline_qc[cols_front + [c for c in pipeline_qc.columns if c not in cols_front]]

data_qc_path = OUT_DIR / "dataQC_with_identity.csv"
pipeline_qc_path = OUT_DIR / "pipelineQC_with_identity.csv"
data_qc.to_csv(data_qc_path, index=False)
pipeline_qc.to_csv(pipeline_qc_path, index=False)
print("Đã lưu:", data_qc_path)
print("Đã lưu:", pipeline_qc_path)
data_qc.head()

## 4. Tổng quan QC

In [ ]:
summary_cols = ["File_Length_s", "Pct_Epochs_Retained", "N_Epochs_Post_Rejection"]
data_qc[summary_cols].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(data_qc["Pct_Epochs_Retained"].dropna(), bins=20, color="#0d7c76", edgecolor="white")
axes[0].axvline(data_qc["Pct_Epochs_Retained"].median(), color="black", linestyle="--", linewidth=1,
                label=f"trung vị = {data_qc['Pct_Epochs_Retained'].median():.1f}%")
axes[0].set_xlabel("% epoch giữ lại / session")
axes[0].set_ylabel("Số session")
axes[0].set_title("Phân bố tỉ lệ epoch giữ lại")
axes[0].legend()

order = data_qc.groupby("participant_key")["Pct_Epochs_Retained"].median().sort_values().index
data_qc.boxplot(column="Pct_Epochs_Retained", by="participant_key", ax=axes[1],
                positions=range(len(order)))
axes[1].set_xticklabels(order, rotation=90)
axes[1].set_title("% epoch giữ lại theo participant")
axes[1].set_xlabel("")
plt.suptitle("")
fig.tight_layout()
fig.savefig(OUT_DIR / "qc_epoch_retention_overview.png", dpi=130)
plt.show()

In [ ]:
coh_col = [c for c in pipeline_qc.columns if c.startswith("lineNoise_coherence_") and "50" in c]
fig, ax = plt.subplots(figsize=(7, 4.5))
for c in sorted(coh_col):
    ax.hist(pipeline_qc[c].dropna(), bins=20, alpha=0.6, label=c.replace("lineNoise_coherence_", ""))
ax.set_xlabel("Coherence (0-1, càng thấp = notch loại càng sạch)")
ax.set_ylabel("Số session")
ax.set_title("Coherence trước/sau notch tại các tần số quan tâm (50/100Hz ± lân cận)")
ax.legend(title="Tần số")
fig.tight_layout()
fig.savefig(OUT_DIR / "qc_linenoise_coherence.png", dpi=130)
plt.show()

print(pipeline_qc[[c for c in pipeline_qc.columns if c.startswith("lineNoise_coherence_")]].median())

## 5. So sánh trước/sau áp dụng pipeline (notch + bandpass)

"Trước" = tín hiệu thô (đã quy đổi Volt, chưa xử lý gì). "Sau" = sau
notch 50Hz + bandpass 1-45Hz (tín hiệu liên tục, TRƯỚC khi segment/reject).
Đối lập hoàn toàn với wavelet (xoá phần lớn công suất mọi băng) — notch+bandpass
chỉ cắt đúng phần ngoài 1-45Hz và đúng tần số điện lưới, phần còn lại giữ
nguyên gần như 100%.

In [ ]:
bp_cols_retained = [f"{k}_power_retained_pct" for k in BANDS]
print("% công suất còn lại SAU notch+bandpass so với TRƯỚC xử lý (trung vị các session):")
print(pipeline_qc[bp_cols_retained].median())
print()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
band_names = list(BANDS.keys())
medians = [pipeline_qc[f"{k}_power_retained_pct"].median() for k in band_names]
bars = ax.bar(band_names, medians, color="#0d7c76")
ax.axhline(100, color="gray", linestyle="--", linewidth=1, label="100% (không mất gì)")
ax.set_ylabel("% công suất còn lại sau notch+bandpass")
ax.set_title("Công suất theo băng tần: trước vs sau (trung vị các session)")
ax.set_ylim(0, 115)
for b, v in zip(bars, medians):
    ax.text(b.get_x() + b.get_width()/2, v + 2, f"{v:.0f}%", ha="center", fontsize=10)
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "qc_band_power_before_after.png", dpi=130)
plt.show()

In [ ]:
# PSD trung bình toàn bộ session, trước và sau -- minh hoạ notch cắt đúng 50/100Hz,
# bandpass cắt đúng ngoài 1-45Hz, phần còn lại giữ nguyên hình dạng phổ.
example_stem = sorted(psd_examples.keys())[0]
f_before, psd_before, f_after, psd_after = psd_examples[example_stem]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.semilogy(f_before, psd_before, label="Trước xử lý", color="#a8402a", alpha=0.85)
ax.semilogy(f_after, psd_after, label="Sau notch+bandpass", color="#0d7c76", alpha=0.9)
ax.axvline(50, color="gray", linestyle=":", linewidth=1)
ax.axvline(1, color="gray", linestyle=":", linewidth=1)
ax.axvline(45, color="gray", linestyle=":", linewidth=1)
ax.set_xlim(0, 100)
ax.set_xlabel("Tần số (Hz)")
ax.set_ylabel("PSD (V²/Hz, log scale)")
ax.set_title(f"Ví dụ PSD trước/sau — {example_stem}")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "qc_psd_example_before_after.png", dpi=130)
plt.show()

## 6. Tổng kết & đường dẫn output

- `data/meditation_ingestion_output/analytics/eeg_qc/dataQC_with_identity.csv` — QC từng session (độ dài, epoch pre/post/% giữ lại)
- `data/meditation_ingestion_output/analytics/eeg_qc/pipelineQC_with_identity.csv` — QC từng bước (coherence line-noise, công suất theo băng tần trước/sau)
- `data/meditation_ingestion_output/analytics/eeg_qc/run_config.yaml` — cấu hình đã dùng
- `data/meditation_ingestion_output/analytics/eeg_qc/epochs_processed/` — epoch đã xử lý (`-epo.fif`), dùng cho phân tích spectral tiếp theo
- `data/meditation_ingestion_output/analytics/eeg_qc/qc_*.png` — biểu đồ QC + so sánh trước/sau

**`notebooks/eeg_happe_qc.ipynb`** (pipeline HAPPE đầy đủ, có wavelet)
vẫn giữ nguyên làm tham chiếu/so sánh — không dùng cho phân tích cuối.

In [ ]:
print("Output directory:", OUT_DIR)
for p in sorted(OUT_DIR.glob("*")):
    print(" -", p.name)